In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
import os
from transformers import T5Tokenizer, T5ForConditionalGeneration
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/news_summary.csv")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [ ]:
class CustomDataset(Dataset):
  def __init__(self, dataframe, tokenizer):
    self.df = dataframe
    self.tokenizer = tokenizer
    self.source = self.df.text
    self.target = self.df.headlines

  def __len__(self):
    return len(self.df.text)

  def __getitem__(self, idx):
    source_text = self.source[idx]
    target_text = self.target[idx]

    # Clean whitespace
    source_text = " ".join(source_text.split())
    target_text = " ".join(target_text.split())

    # Tokenize both source and target
    source = tokenizer.batch_encode_plus([source_text], max_length=512, pad_to_max_length=True, padding="max_length", return_tensors="pt")
    target = tokenizer.batch_encode_plus([target_text], max_length=50, pad_to_max_length=True, padding="max_length", return_tensors="pt")

    source_ids = source['input_ids'].squeeze(0)
    source_mask = source['attention_mask'].squeeze(0)
    target_ids = target['input_ids'].squeeze(0)
    target_mask = target['attention_mask'].squeeze(0)

    return {
        "source_ids": source_ids.to(dtype=torch.long),
        "source_mask": source_mask.to(dtype=torch.long),
        "target_ids" : target_ids.to(dtype=torch.long),
    }

In [ ]:
# Rephrasing the input text
df['text'] = "Summarize: " + df['text']

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-base")

In [ ]:
train_dataset = df.sample(100, random_state=42)
train_dataset = train_dataset.reset_index(drop=True)
training_set = CustomDataset(train_dataset, tokenizer)
train_loader = DataLoader(training_set, batch_size=8, shuffle=True)

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-base")
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
model.train()

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [ ]:
for _, data in enumerate(train_loader, 0):
  y = data['target_ids'].to(device, dtype=torch.long)
  y_ids = y[:,:-1].contiguous()
  lm_labels = y[:,:-1].clone().detach()
  lm_labels[y[:,:-1] == tokenizer.pad_token_id] = -100
  ids = data['source_ids'].to(device, dtype=torch.long)
  mask = data['source_mask'].to(device, dtype=torch.long)

  outputs = model(input_ids=ids, attention_mask=mask, decoder_input_ids=y_ids, labels=lm_labels)
  loss = outputs[0]

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

## Testing the model

In [ ]:
example = """
Vincent van Gogh (1853–1890) was a Dutch Post-Impressionist painter known for his expressive brushwork and vivid colors. Initially working as an art dealer and a preacher, he turned to painting in his late 20s. Struggling with mental illness and poverty, he created over 2,000 artworks, including *Starry Night* and *Sunflowers*. His unique style was underappreciated during his lifetime, and he sold only a few paintings. Van Gogh famously cut off part of his ear after a dispute with fellow artist Paul Gauguin. He spent time in asylums before tragically dying from a gunshot wound, believed to be self-inflicted. His work later became widely celebrated, influencing modern art profoundly. Summarize: """

In [ ]:
example_encoded = tokenizer.encode(example, return_tensors="pt").to(device)
generated = model.generate(example_encoded)

In [ ]:
tokenizer.decode(generated[0], skip_special_tokens=True)

'van Gogh (1853–1890) was a Dutch post-impressionist painter'

In [ ]:
torch.save(model.state_dict(), "./model.pth")
model.save_pretrained("t5_finetuned")
tokenizer.save_pretrained("t5_finetuned")

('t5_finetuned/tokenizer_config.json',
 't5_finetuned/special_tokens_map.json',
 't5_finetuned/spiece.model',
 't5_finetuned/added_tokens.json')